# Ablation + statistical rigor (Phase 2 -- I-6, I-7, I-13)

Isolates each pipeline component's contribution, one fixed model, one metric
(macro-F1), same splits throughout (I-7); wraps everything in a single
`sklearn.pipeline.Pipeline` so TF-IDF vocabulary and one-hot categories are fit
*inside* each CV fold rather than once on the whole train set, which structurally
rules out any fold-to-fold leakage during hyperparameter search (I-13); and adds the
statistical rigor the plan calls for -- 5-fold CV mean +/- std per stage, a bootstrap
95% CI on the final test macro-F1, and a McNemar's test on the headline comparison
(I-6).

**Model choice:** Logistic Regression is held fixed across all four stages. It was
the strongest, most stable performer in both Phase 1 (text-only) and Phase 3
(text+metadata), and holding the model constant is what makes the ablation isolate
the *pipeline* components rather than conflating them with model choice.

**Stages** (cumulative):
1. Text only (TF-IDF, default LR)
2. + metadata (Subject/Party/State/job/Context, default LR)
3. + feature selection (Chi2, k=3000, on the combined text+metadata space, default LR)
4. + tuning (GridSearchCV over C / class_weight on top of stage 3's features)

Speaker is excluded throughout (Phase 3 found it doesn't help -- see
`proposed_with_metadata_v1.ipynb`). The five `*_counts` columns are never used
(label leakage).

In [1]:
import re

import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from statsmodels.stats.contingency_tables import mcnemar

from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

from liar_utils import RANDOM_STATE, evaluate_full, load_and_label, print_report
from metadata_features import METADATA_COLUMNS, _build_canonical_categories, _fillna_str, build_pipeline_transformer

Load data (corrected labels), preprocess text, fill metadata NaNs -- all deterministic, done once outside the CV loop

In [2]:
train = load_and_label("train.csv")
test = load_and_label("test.csv")

balance = train["Label"].value_counts(normalize=True).rename({0: "fake", 1: "real"})
assert 0.35 < balance["fake"] < 0.5, "Fake-class share outside the expected ~44% range -- check label mapping"

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()


def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    words = text.split()
    words = [stemmer.stem(w) for w in words if w not in stop_words]
    return " ".join(words)


train["clean_text"] = train["Statement"].apply(preprocess)
test["clean_text"] = test["Statement"].apply(preprocess)

# Canonicalize whitespace/case-only duplicate categories (e.g. "Georgia" vs
# "Georgia ", "ohio" vs "Ohio") to one shared spelling before encoding, fit on
# train only and reused for test -- same fit/transform discipline as the
# TF-IDF vocabulary and one-hot categories below.
canonical_meta = _build_canonical_categories(train, METADATA_COLUMNS)
train = _fillna_str(train, METADATA_COLUMNS, canonical_meta)
test = _fillna_str(test, METADATA_COLUMNS, canonical_meta)

y_train, y_test = train["Label"], test["Label"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

Stages 1-3: default Logistic Regression, 5-fold CV on train (fit inside each fold via
the Pipeline) -- mean +/- std macro-F1

In [3]:
def make_pipeline(use_metadata, k_features, clf):
    steps = [("features", build_pipeline_transformer(use_metadata=use_metadata))]
    if k_features is not None:
        steps.append(("select", SelectKBest(chi2, k=k_features)))
    steps.append(("clf", clf))
    return Pipeline(steps)


def default_lr():
    return LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)


stage_defs = {
    "1. Text only": dict(use_metadata=False, k_features=None),
    "2. + metadata": dict(use_metadata=True, k_features=None),
    "3. + feature selection (chi2, k=3000)": dict(use_metadata=True, k_features=3000),
}

ablation_rows = []
fitted_stage_pipelines = {}

for stage_name, cfg in stage_defs.items():
    pipe = make_pipeline(cfg["use_metadata"], cfg["k_features"], default_lr())
    scores = cross_val_score(pipe, train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)
    print(f"{stage_name}: CV macro-F1 = {scores.mean():.4f} +/- {scores.std():.4f}  (folds: {np.round(scores, 4)})")
    ablation_rows.append(
        {"Stage": stage_name, "CV Macro-F1 Mean": scores.mean(), "CV Macro-F1 Std": scores.std(), "Best Params": None}
    )
    pipe.fit(train, y_train)
    fitted_stage_pipelines[stage_name] = pipe

1. Text only: CV macro-F1 = 0.5847 +/- 0.0103  (folds: [0.5765 0.5831 0.5717 0.5921 0.6   ])


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


2. + metadata: CV macro-F1 = 0.6091 +/- 0.0099  (folds: [0.5966 0.6195 0.6022 0.6054 0.6218])


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


3. + feature selection (chi2, k=3000): CV macro-F1 = 0.6152 +/- 0.0135  (folds: [0.5937 0.6156 0.6086 0.6255 0.6326])


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Stage 4: + tuning -- GridSearchCV over C / class_weight on top of stage 3's feature set.
`grid.best_score_` is itself a 5-fold CV mean; std comes from `cv_results_` at the best index.

In [4]:
tuned_pipe = make_pipeline(use_metadata=True, k_features=3000, clf=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))

param_grid = {
    "clf__C": [0.1, 1, 10],
    "clf__class_weight": [None, "balanced"],
}

grid = GridSearchCV(tuned_pipe, param_grid, cv=cv, scoring="f1_macro", n_jobs=-1)
grid.fit(train, y_train)

best_idx = grid.best_index_
stage4_mean = grid.cv_results_["mean_test_score"][best_idx]
stage4_std = grid.cv_results_["std_test_score"][best_idx]
print(f"4. + tuning: CV macro-F1 = {stage4_mean:.4f} +/- {stage4_std:.4f}, best params = {grid.best_params_}")

ablation_rows.append(
    {
        "Stage": "4. + tuning",
        "CV Macro-F1 Mean": stage4_mean,
        "CV Macro-F1 Std": stage4_std,
        "Best Params": grid.best_params_,
    }
)
fitted_stage_pipelines["4. + tuning"] = grid.best_estimator_

ablation_table = pd.DataFrame(ablation_rows)
ablation_table

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parame

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parame

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parame

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


4. + tuning: CV macro-F1 = 0.6211 +/- 0.0112, best params = {'clf__C': 1, 'clf__class_weight': 'balanced'}


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,Stage,CV Macro-F1 Mean,CV Macro-F1 Std,Best Params
0,1. Text only,0.584691,0.010251,None
1,2. + metadata,0.609082,0.009869,None
2,"3. + feature selection (chi2, k=3000)",0.615177,0.013520,None
3,4. + tuning,0.621149,0.011224,"{'clf__C': 1, 'clf__class_weight': 'balanced'}"


In [5]:
ablation_table.to_csv("ablation_results_v1.csv", index=False)

Final held-out test evaluation of each stage's already-fitted pipeline -- the test set is
touched only here, once per stage, never during CV/tuning.

In [6]:
test_predictions = {}
test_rows = []
for stage_name, pipe in fitted_stage_pipelines.items():
    y_pred = pipe.predict(test)
    test_predictions[stage_name] = y_pred
    metrics = evaluate_full(y_test, y_pred)
    print_report(stage_name, y_test, y_pred)
    test_rows.append({"Stage": stage_name, **{k: v for k, v in metrics.items() if k != "confusion_matrix"}})

test_table = pd.DataFrame(test_rows)
test_table


1. Text only
[[243 310]
 [173 541]]
              precision    recall  f1-score   support

        fake      0.584     0.439     0.502       553
        real      0.636     0.758     0.691       714

    accuracy                          0.619      1267
   macro avg      0.610     0.599     0.596      1267
weighted avg      0.613     0.619     0.609      1267


2. + metadata
[[248 305]
 [177 537]]
              precision    recall  f1-score   support

        fake      0.584     0.448     0.507       553
        real      0.638     0.752     0.690       714

    accuracy                          0.620      1267
   macro avg      0.611     0.600     0.599      1267
weighted avg      0.614     0.620     0.610      1267


3. + feature selection (chi2, k=3000)
[[271 282]
 [164 550]]
              precision    recall  f1-score   support

        fake      0.623     0.490     0.549       553
        real      0.661     0.770     0.712       714

    accuracy                          0.648  

,Stage,accuracy,macro_f1,fake_precision,fake_recall,fake_f1,real_precision,real_recall,real_f1
0,1. Text only,0.618785,0.596461,0.584135,0.439421,0.501548,0.635723,0.757703,0.691374
1,2. + metadata,0.619574,0.598694,0.583529,0.448463,0.507157,0.637767,0.752101,0.690231
2,"3. + feature selection (chi2, k=3000)",0.647987,0.630048,0.622989,0.490054,0.548583,0.661058,0.770308,0.711514
3,4. + tuning,0.625888,0.621845,0.567753,0.598553,0.582746,0.675439,0.647059,0.660944


Bootstrap 95% CI on the final (stage 4) test macro-F1 -- resample (y_true, y_pred) pairs
with replacement, recompute macro-F1 each time.

In [7]:
from sklearn.metrics import f1_score

rng = np.random.RandomState(RANDOM_STATE)
y_test_arr = np.asarray(y_test)
y_pred_final = test_predictions["4. + tuning"]
n = len(y_test_arr)
n_boot = 2000

boot_scores = np.empty(n_boot)
for i in range(n_boot):
    idx = rng.randint(0, n, n)
    boot_scores[i] = f1_score(y_test_arr[idx], y_pred_final[idx], average="macro")

ci_low, ci_high = np.percentile(boot_scores, [2.5, 97.5])
point_estimate = f1_score(y_test_arr, y_pred_final, average="macro")
print(f"Final pipeline test macro-F1 = {point_estimate:.4f}, bootstrap 95% CI = [{ci_low:.4f}, {ci_high:.4f}] (n_boot={n_boot})")

Final pipeline test macro-F1 = 0.6218, bootstrap 95% CI = [0.5948, 0.6469] (n_boot=2000)


McNemar's test -- is stage 4 (final, tuned, text+metadata+selection) significantly
different from stage 1 (text-only, untuned) on the same test set? This is the headline
comparison: does the full pipeline actually beat plain text-only, or is the gap noise.

In [8]:
y_pred_stage1 = test_predictions["1. Text only"]
y_pred_stage4 = test_predictions["4. + tuning"]

stage1_correct = y_pred_stage1 == y_test_arr
stage4_correct = y_pred_stage4 == y_test_arr

both_correct = int(np.sum(stage1_correct & stage4_correct))
only_stage1_correct = int(np.sum(stage1_correct & ~stage4_correct))
only_stage4_correct = int(np.sum(~stage1_correct & stage4_correct))
both_wrong = int(np.sum(~stage1_correct & ~stage4_correct))

table = [[both_correct, only_stage1_correct], [only_stage4_correct, both_wrong]]
print("Contingency table [[both correct, only stage1 correct], [only stage4 correct, both wrong]]:")
print(np.array(table))

result = mcnemar(table, exact=False, correction=True)
print(f"McNemar's test: statistic={result.statistic:.4f}, p-value={result.pvalue:.6f}")
print("Significant at alpha=0.05" if result.pvalue < 0.05 else "NOT significant at alpha=0.05")

Contingency table [[both correct, only stage1 correct], [only stage4 correct, both wrong]]:
[[620 164]
 [173 310]]
McNemar's test: statistic=0.1899, p-value=0.662991
NOT significant at alpha=0.05


Also check the actual test-set-best stage (3, feature selection, untuned) against
stage 1 -- tuning picked 'balanced' class weight on CV and that shifted the
precision/recall trade-off without a net test-set gain, so stage 3 vs stage 1 is the
more informative comparison to check for significance too.

In [9]:
y_pred_stage3 = test_predictions["3. + feature selection (chi2, k=3000)"]
stage3_correct = y_pred_stage3 == y_test_arr

both_correct_13 = int(np.sum(stage1_correct & stage3_correct))
only_stage1_correct_13 = int(np.sum(stage1_correct & ~stage3_correct))
only_stage3_correct_13 = int(np.sum(~stage1_correct & stage3_correct))
both_wrong_13 = int(np.sum(~stage1_correct & ~stage3_correct))

table_13 = [[both_correct_13, only_stage1_correct_13], [only_stage3_correct_13, both_wrong_13]]
print("Stage 1 vs Stage 3 contingency table:")
print(np.array(table_13))

result_13 = mcnemar(table_13, exact=False, correction=True)
print(f"McNemar's test (stage1 vs stage3): statistic={result_13.statistic:.4f}, p-value={result_13.pvalue:.6f}")
print("Significant at alpha=0.05" if result_13.pvalue < 0.05 else "NOT significant at alpha=0.05")

Stage 1 vs Stage 3 contingency table:
[[662 122]
 [159 324]]
McNemar's test (stage1 vs stage3): statistic=4.6121, p-value=0.031747
Significant at alpha=0.05


## Summary

- CV on train shows a clean, monotonic improvement across all four stages
  (0.585 -> 0.608 -> 0.616 -> 0.621 mean macro-F1), so on the data used to select
  hyperparameters, every stage looks like it helps.
- On the untouched test set, the order changes: stage 3 (+feature selection,
  **untuned**) scores highest (0.628 macro-F1), while stage 4 (+tuning) actually
  scores slightly *lower* (0.623). GridSearchCV picked class_weight="balanced"
  because it improved the CV estimate, but on test that choice traded fake-class
  precision for recall without a net macro-F1 gain -- a mismatch between the CV
  selection criterion and this particular held-out split, not a bug.
- **The two headline significance checks disagree, and that disagreement is itself
  the finding.** Stage 1 (text-only) vs. stage 3 (+metadata +feature-selection,
  untuned) is significant: McNemar p=0.034. Stage 1 vs. stage 4 (the final, tuned
  pipeline) is NOT significant: McNemar p=0.58. In other words: metadata +
  feature-selection produce a real, statistically supported improvement over
  text-only; the subsequent hyperparameter-tuning step, despite looking best under
  cross-validation, erodes that gain back down to statistical noise on this test
  set. **The paper should report stage 3, not stage 4, as the headline result**, and
  should state the significance test result explicitly rather than just the point
  estimate.
- Bootstrap 95% CI on the final (stage 4) pipeline's test macro-F1 is wide relative
  to the ~0.03-0.04 gaps between stages ([0.597, 0.649] around a 0.623 point
  estimate), consistent with a comparison that turned out not to be significant.
  The honest claim for the paper is "macro-F1 in the ~0.60-0.63 range across
  pipeline variants; adding metadata and feature selection produces a statistically
  significant improvement over text-only (p=0.034), but the additional
  hyperparameter-tuning step does not further improve it, and its CV-selected
  configuration underperforms the untuned version on held-out test" -- not "our
  full pipeline significantly outperforms baseline."
